In [61]:
# ============================================================
# Block 1 — Setup and Load Data
# ============================================================

import numpy as np
import pandas as pd
import json
import gc
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

emb_cols = [f'emb_{i}' for i in range(32)]

print("=== Stage 2 — Classifier ===")

# ── Load features ─────────────────────────────────────────
train_features = pd.read_csv('ori_feature/train_features_v631.csv')
test_features  = pd.read_csv('ori_feature/test_features_v631.csv')

# load feature cols
with open('feature_cols.json', 'r') as f:
    feature_cols = json.load(f)

print(f"Train shape:  {train_features.shape}")
print(f"Test shape:   {test_features.shape}")
print(f"Feature cols: {len(feature_cols)}")
print(f"Fraud rate:   {train_features['isFraud'].mean():.2%}")

=== Stage 2 — Classifier ===
Train shape:  (590540, 638)
Test shape:   (506691, 637)
Feature cols: 631
Fraud rate:   3.50%


In [82]:
# load embeddings
# compare OOF vs test embedding distributions
oof_emb_df  = pd.read_csv('/workspace/Embedding_res_chain/oof_embeddings.csv', index_col=0)
test_emb_df = pd.read_csv('/workspace/Embedding_res_chain/test_embeddings_sliding_chain_5.csv', index_col=0)

print(f"OOF embeddings:  {oof_emb_df.shape}")
print(f"Test embeddings: {test_emb_df.shape}")

OOF embeddings:  (295270, 32)
Test embeddings: (506691, 32)


In [83]:
oof_stats = {
    'mean': oof_emb_df[emb_cols].mean().mean(),
    'std':  oof_emb_df[emb_cols].std().mean(),
    'max':  oof_emb_df[emb_cols].max().max(),
    'min':  oof_emb_df[emb_cols].min().min(),
}

test_stats = {
    'mean': test_emb_df[emb_cols].mean().mean(),
    'std':  test_emb_df[emb_cols].std().mean(),
    'max':  test_emb_df[emb_cols].max().max(),
    'min':  test_emb_df[emb_cols].min().min(),
}

print("OOF embeddings:", oof_stats)
print("Test embeddings:", test_stats)

# split test embeddings by chunk (which window generated them)
n_test = len(test_emb_df)
chunk_size = n_test // 10

for i in range(10):
    start = i * chunk_size
    end = (i+1) * chunk_size if i < 9 else n_test
    chunk = test_emb_df.iloc[start:end]
    print(f"Test chunk {i+1}: "
          f"mean={chunk.values.mean():.4f}, "
          f"std={chunk.values.std():.4f}")

OOF embeddings: {'mean': np.float64(0.2324504477320985), 'std': np.float64(0.49540702486180654), 'max': np.float64(62.800537), 'min': np.float64(0.0)}
Test embeddings: {'mean': np.float64(0.135743638833379), 'std': np.float64(0.2691885836959631), 'max': np.float64(71.35598), 'min': np.float64(0.0)}
Test chunk 1: mean=0.1225, std=0.3946
Test chunk 2: mean=0.1181, std=0.4371
Test chunk 3: mean=0.1284, std=0.4453
Test chunk 4: mean=0.1198, std=0.3876
Test chunk 5: mean=0.1435, std=0.4626
Test chunk 6: mean=0.1286, std=0.4144
Test chunk 7: mean=0.1470, std=0.5025
Test chunk 8: mean=0.1449, std=0.6060
Test chunk 9: mean=0.1651, std=0.6849
Test chunk 10: mean=0.1396, std=0.5077


In [84]:
# ============================================================
# Block 2 — Load Embeddings and Create Folds
# ============================================================



# sort train by time
train_sorted = train_features.sort_values('TransactionDT').reset_index(drop=True)

# create folds
n = len(train_sorted)
fold_size = n // 10
folds = {}
for i in range(10):
    start = i * fold_size
    end   = (i+1) * fold_size if i < 9 else n
    folds[i+1] = train_sorted.iloc[start:end].index.tolist()

# T6~T9 train, T10 val
T6_T9_idx = sum([folds[i] for i in range(6, 10)], [])
T10_idx   = folds[10]

# filter to rows with OOF embeddings
T6_T9_in_oof = [i for i in T6_T9_idx if i in oof_emb_df.index]
T10_in_oof   = [i for i in T10_idx   if i in oof_emb_df.index]

print(f"\nTrain (T6~T9): {len(T6_T9_in_oof):,}")
print(f"Val   (T10):   {len(T10_in_oof):,}")


Train (T6~T9): 236,216
Val   (T10):   59,054


In [85]:
# ============================================================
# Block 3 — Prepare Feature Matrices
# ============================================================

# ── Tabular features ──────────────────────────────────────
X_train_tab = train_sorted.loc[T6_T9_in_oof, feature_cols]
X_val_tab = train_sorted.loc[T10_in_oof, feature_cols]
X_test_tab = test_features[feature_cols]

# ── GCN embeddings ────────────────────────────────────────
X_train_emb = oof_emb_df.loc[T6_T9_in_oof, emb_cols].reset_index(drop=True)
X_val_emb   = oof_emb_df.loc[T10_in_oof,   emb_cols].reset_index(drop=True)
X_test_emb  = test_emb_df[emb_cols].reset_index(drop=True)

# ── Labels ────────────────────────────────────────────────
y_train = train_sorted.loc[T6_T9_in_oof, 'isFraud'].reset_index(drop=True)
y_val   = train_sorted.loc[T10_in_oof,   'isFraud'].reset_index(drop=True)

print(f"Fraud rate train: {y_train.mean():.2%}")
print(f"Fraud rate val:   {y_val.mean():.2%}")

# ── Combine tabular + embeddings ──────────────────────────
X_train = pd.concat([
    X_train_tab.reset_index(drop=True),
    X_train_emb
], axis=1)

X_val = pd.concat([
    X_val_tab.reset_index(drop=True),
    X_val_emb
], axis=1)

X_test = pd.concat([
    X_test_tab.reset_index(drop=True),
    X_test_emb
], axis=1)

print(f"\nX_train: {X_train.shape}")
print(f"X_val:   {X_val.shape}")
print(f"X_test:  {X_test.shape}")

Fraud rate train: 3.62%
Fraud rate val:   3.75%

X_train: (236216, 663)
X_val:   (59054, 663)
X_test:  (506691, 663)


In [86]:
cat_fea = ['ProductCD', 'card3', 'card4', 'card5', 'card6', 'addr2', 'P_emaildomain', 'R_emaildomain', 'DeviceType', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_13', 'id_14', 'id_15', 'id_17', 'id_18', 'id_22', 'id_23', 'id_24', 'id_26', 'id_28', 'id_30', 'id_31', 'id_32', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'amt_decimal', 'amt_last_digit', 'D15_bin']

for col in list(cat_fea):
    tr_cats = X_train_tab[col].astype('category').cat.categories
    for df in [X_train_tab,X_val_tab,X_test,X_test_tab,X_train,X_val]:# 检查特征的方差
        df[col] = pd.Categorical(df[col], categories=tr_cats)

In [67]:
# ============================================================
# Block 4 — Baseline: Tabular Only
# ============================================================

print("=== Baseline: Tabular Only ===")

baseline_model = lgb.LGBMClassifier(
    n_estimators=1000, learning_rate=0.01, num_leaves=64,
    random_state=42, n_jobs=-1, verbosity=-1
)
baseline_model.fit(
    X_train_tab.reset_index(drop=True), y_train,
    eval_set  = [(X_val_tab.reset_index(drop=True), y_val)],
    eval_metric = 'auc',
    categorical_feature=cat_fea,
callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(100)]
)


baseline_pred = baseline_model.predict_proba(
    X_val_tab.reset_index(drop=True)
)[:, 1]
baseline_auc  = roc_auc_score(y_val, baseline_pred)

print(f"\n✅ Baseline AUC (tabular only): {baseline_auc:.4f}")

=== Baseline: Tabular Only ===
[100]	valid_0's auc: 0.894136	valid_0's binary_logloss: 0.107735
[200]	valid_0's auc: 0.913052	valid_0's binary_logloss: 0.0966176
[300]	valid_0's auc: 0.926206	valid_0's binary_logloss: 0.0903708
[400]	valid_0's auc: 0.934608	valid_0's binary_logloss: 0.0864294
[500]	valid_0's auc: 0.938438	valid_0's binary_logloss: 0.0842233
[600]	valid_0's auc: 0.940495	valid_0's binary_logloss: 0.0830714
[700]	valid_0's auc: 0.940926	valid_0's binary_logloss: 0.0829372

✅ Baseline AUC (tabular only): 0.9410


In [72]:

########## filter feature for basline model 
import shap

explainer = shap.TreeExplainer(baseline_model)
shap_values = explainer.shap_values(X_train_tab.sample(10000))

print(f"Mean abs SHAP for all feature: {abs(shap_values).mean():.4f}")

# get per-feature SHAP magnitudes
shap_per_feature = abs(shap_values).mean(axis=0)

# rank embedding features by SHAP
all_shap_df = pd.DataFrame({
    'feature': X_train_tab.columns,
    'shap':    shap_per_feature
})

all_shap_df = all_shap_df.sort_values('shap', ascending=False)

useful_cols = all_shap_df.head(400)['feature'].tolist()
X_train_tab_v2 = X_train_tab[useful_cols]
X_val_tab_v2   = X_val_tab[useful_cols]


shap_cat_fea = []
for i in cat_fea:
    if i in useful_cols:
        shap_cat_fea.append(i)


# retrain
tab_model_v2 = lgb.LGBMClassifier(
    n_estimators=1000, learning_rate=0.01, num_leaves=64,
    random_state=42, n_jobs=-1, verbosity=-1
)

tab_model_v2.fit(
    X_train_tab_v2, y_train,
    eval_set    = [(X_val_tab_v2, y_val)],
    eval_metric = 'auc',
    categorical_feature=shap_cat_fea,
     callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(100)]
)

tab_pred = tab_model_v2.predict_proba(X_val_tab_v2)[:, 1]
tab_auc  = roc_auc_score(y_val, tab_pred)

print(f"\n✅ Full AUC (tabular ): {tab_auc:.4f}")


[100]	valid_0's auc: 0.893458	valid_0's binary_logloss: 0.107903
[200]	valid_0's auc: 0.913404	valid_0's binary_logloss: 0.0967188
[300]	valid_0's auc: 0.925627	valid_0's binary_logloss: 0.0904773
[400]	valid_0's auc: 0.933881	valid_0's binary_logloss: 0.0867013
[500]	valid_0's auc: 0.938374	valid_0's binary_logloss: 0.0843189
[600]	valid_0's auc: 0.940361	valid_0's binary_logloss: 0.0831823
[700]	valid_0's auc: 0.940983	valid_0's binary_logloss: 0.0829536

✅ Full AUC (tabular ): 0.9411


In [17]:
# ── Generate submission ───────────────────────────────────
print("\n=== Generating Submission ===")

test_pred = baseline_model.predict_proba(X_test_tab)[:, 1]

submission = pd.DataFrame({
    'TransactionID': test_features['TransactionID'],
    'isFraud':       test_pred
})

submission.to_csv('/workspace/submission_tabular_half.csv', index=False)

print(f"Submission shape: {submission.shape}")
print(f"Fraud rate pred:  {test_pred.mean():.4f}")
print(f"Min: {test_pred.min():.4f} | Max: {test_pred.max():.4f}")


=== Generating Submission ===
Submission shape: (506691, 2)
Fraud rate pred:  0.0268
Min: 0.0005 | Max: 0.9896

✅ Saved: /workspace/submission_tabular_full.csv


NameError: name 'val_auc' is not defined

In [87]:
# ============================================================
# Block 5 — Full Model: Tabular + GCN Embeddings
# ============================================================

print("=== Full Model: Tabular + GCN Embeddings ===")

full_model = lgb.LGBMClassifier(
    n_estimators=1000, learning_rate=0.01, num_leaves=64,
    random_state=42, n_jobs=-1, verbosity=-1
)

full_model.fit(
    X_train, y_train,
    eval_set    = [(X_val, y_val)],
    eval_metric = 'auc',
    categorical_feature=cat_fea,
     callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(100)]
)

full_pred = full_model.predict_proba(X_val)[:, 1]
full_auc  = roc_auc_score(y_val, full_pred)

print(f"\n✅ Full AUC (tabular + GCN): {full_auc:.4f}")
print(f"   Baseline AUC (tabular):   {baseline_auc:.4f}")
print(f"   Improvement:              +{full_auc - baseline_auc:.4f}")

=== Full Model: Tabular + GCN Embeddings ===
[100]	valid_0's auc: 0.911563	valid_0's binary_logloss: 0.109241
[200]	valid_0's auc: 0.924369	valid_0's binary_logloss: 0.097738
[300]	valid_0's auc: 0.932906	valid_0's binary_logloss: 0.0918187
[400]	valid_0's auc: 0.938534	valid_0's binary_logloss: 0.0881729
[500]	valid_0's auc: 0.94165	valid_0's binary_logloss: 0.0861355
[600]	valid_0's auc: 0.942598	valid_0's binary_logloss: 0.0852893

✅ Full AUC (tabular + GCN): 0.9427
   Baseline AUC (tabular):   0.9410
   Improvement:              +0.0016


In [88]:
# ============================================================
# Block 6 — Feature Importance
# ============================================================

print("=== Feature Importance ===")

importance_df = pd.DataFrame({
    'feature': X_train.columns.tolist(),
    'importance': full_model.feature_importances_
}).sort_values('importance', ascending=False)

# how many embedding features in top 50?
emb_in_top50 = importance_df.head(50)['feature'].str.startswith('emb_').sum()
print(f"Embedding features in top 50: {emb_in_top50}/32")

print("\nTop 20 features:")
print(importance_df.head(20).to_string(index=False))

# average importance of embeddings vs tabular
emb_imp = importance_df[importance_df['feature'].str.startswith('emb_')]['importance'].mean()
tab_imp = importance_df[~importance_df['feature'].str.startswith('emb_')]['importance'].mean()
print(f"Avg emb importance: {emb_imp:.2f}")
print(f"Avg tab importance: {tab_imp:.2f}")

=== Feature Importance ===
Embedding features in top 50: 6/32

Top 20 features:
                    feature  importance
                amt_decimal        3105
user_id1_TransactionAmt_sum         579
                      id_31         565
           user_id1_C13_max         548
                         C1         528
           user_id1_C13_sum         518
user_id1_TransactionAmt_max         493
                        C11         448
                     emb_14         438
              P_emaildomain         411
                        C14         408
                    D4_norm         389
                    D1_norm         383
user_id1_TransactionAmt_p25         380
                    D2_norm         361
                        C13         344
user_id1_TransactionAmt_p50         339
                      emb_5         332
         user_id1_dist1_min         323
                     emb_15         313
Avg emb importance: 102.22
Avg tab importance: 53.32


In [89]:
import shap

explainer = shap.TreeExplainer(full_model)
shap_values = explainer.shap_values(X_val.sample(10000))

# look at SHAP values for embedding features
emb_cols_idx = [i for i, c in enumerate(X_val.columns) if c.startswith('emb_')]
emb_shap = shap_values[:, emb_cols_idx]
print(f"Mean abs SHAP for embeddings: {abs(emb_shap).mean():.4f}")

Mean abs SHAP for embeddings: 0.0251


In [95]:
# get per-feature SHAP magnitudes
shap_per_feature = abs(shap_values).mean(axis=0)

# rank embedding features by SHAP
emb_shap_df = pd.DataFrame({
    'feature': X_val.columns,
    'shap':    shap_per_feature
})
emb_shap_df = emb_shap_df[emb_shap_df['feature'].str.startswith('emb_')]
emb_shap_df = emb_shap_df.sort_values('shap', ascending=False)
print(emb_shap_df)

#keep only top 5-10 embeddings
top_emb = emb_shap_df.head(10)['feature'].tolist()
print(f"Keeping: {top_emb}")
#top_emb = ['emb_15','emb_14' ,'emb_18', 'emb_16', 'emb_5', 'emb_2' ,'emb_1']

# rebuild X with reduced embeddings
useful_cols = feature_cols + top_emb
X_train_v2 = X_train[useful_cols]
X_val_v2   = X_val[useful_cols]
X_test_v2  = X_test[useful_cols]

# retrain
full_model_v2 = lgb.LGBMClassifier(
    n_estimators=1000, learning_rate=0.01, num_leaves=64,
    random_state=42, n_jobs=-1, verbosity=-1
)

full_model_v2.fit(
    X_train_v2, y_train,
    eval_set    = [(X_val_v2, y_val)],
    eval_metric = 'auc',
    categorical_feature=cat_fea,
     callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(100)]
)

full_pred = full_model_v2.predict_proba(X_val_v2)[:, 1]
full_auc  = roc_auc_score(y_val, full_pred)

print(f"\n✅ Full AUC (tabular + GCN): {full_auc:.4f}")
print(f"   Baseline AUC (tabular):   {baseline_auc:.4f}")
print(f"   Improvement:              +{full_auc - baseline_auc:.4f}")

    feature      shap
646  emb_15  0.228257
649  emb_18  0.138099
647  emb_16  0.111276
645  emb_14  0.104616
636   emb_5  0.056843
633   emb_2  0.053504
632   emb_1  0.039906
659  emb_28  0.038458
641  emb_10  0.007491
650  emb_19  0.003791
634   emb_3  0.003182
661  emb_30  0.003134
655  emb_24  0.002273
657  emb_26  0.002057
643  emb_12  0.001947
656  emb_25  0.001674
651  emb_20  0.001541
638   emb_7  0.001535
654  emb_23  0.001372
652  emb_21  0.001120
631   emb_0  0.000905
660  emb_29  0.000563
640   emb_9  0.000347
639   emb_8  0.000312
644  emb_13  0.000310
658  emb_27  0.000156
653  emb_22  0.000017
662  emb_31  0.000007
635   emb_4  0.000003
637   emb_6  0.000000
642  emb_11  0.000000
648  emb_17  0.000000
Keeping: ['emb_15', 'emb_18', 'emb_16', 'emb_14', 'emb_5', 'emb_2', 'emb_1', 'emb_28', 'emb_10', 'emb_19']
[100]	valid_0's auc: 0.912359	valid_0's binary_logloss: 0.108308
[200]	valid_0's auc: 0.924208	valid_0's binary_logloss: 0.0968399
[300]	valid_0's auc: 0.932369	valid_

In [90]:
feature_score = pd.DataFrame({
    'feature':       X_val.columns,
    'importance':    full_model.feature_importances_,
    'shap_magnitude': abs(shap_values).mean(axis=0)
})

# normalise both to 0-1 for comparison
feature_score['imp_norm']  = feature_score['importance'] / feature_score['importance'].max()
feature_score['shap_norm'] = feature_score['shap_magnitude'] / feature_score['shap_magnitude'].max()

# combined score
feature_score['quality_score'] = feature_score['imp_norm'] * feature_score['shap_norm']
feature_score = feature_score.sort_values('quality_score', ascending=False)


# for each feature, check sign consistency
shap_signs = np.sign(shap_values)
consistency = abs(shap_signs.mean(axis=0))  
# 1.0 = always same direction, 0.0 = randomly mixed

feature_score['consistency'] = consistency

# with pd.option_context('display.max_columns', None, 'display.width', None):
#     print(feature_score.head(20))
display(feature_score.head(30))

,feature,importance,shap_magnitude,imp_norm,shap_norm,quality_score,consistency
415,amt_decimal,3105,0.114339,1.000000,0.500923,0.500923,0.1986
646,emb_15,313,0.228257,0.100805,1.000000,0.100805,0.0914
18,D1,299,0.163627,0.096296,0.716857,0.069031,0.5308
645,emb_14,438,0.104616,0.141063,0.458327,0.064653,0.8830
649,emb_18,301,0.138099,0.096940,0.605017,0.058651,0.4086
590,user_id1_C13_max,548,0.069949,0.176490,0.306450,0.054085,0.1710
579,user_id1_TransactionAmt_sum,579,0.064104,0.186473,0.280842,0.052370,0.0000
14,C11,448,0.076601,0.144283,0.335590,0.048420,0.8598
17,C14,408,0.076369,0.131401,0.334576,0.043964,0.2766
4,C1,528,0.058826,0.170048,0.257718,0.043824,0.2302


In [ ]:
# ============================================================
# Block 7 — Generate Submission
# ============================================================

print("=== Generating Submission ===")

test_pred = full_model.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({
    'TransactionID': test_features['TransactionID'],
    'isFraud':       test_pred
})

submission.to_csv('/workspace/submission_embeddings_sliding_train_5.csv', index=False)

print(f"Submission shape: {submission.shape}")
print(f"Fraud rate pred:  {test_pred.mean():.4f}")
print(f"Min:              {test_pred.min():.4f}")
print(f"Max:              {test_pred.max():.4f}")

print(f"\n{'='*50}")
print("  Final Results")
print(f"{'='*50}")
print(f"  Full AUC (tabular + GCN):   {full_auc:.4f}")
print(f"  Submission: /workspace/submission_fix_train_5csv ✅")

=== Generating Submission ===
